# Mistral Attempt 01: complete running code

This notebook prepares and can execute200 manager calls with the exact base prompt. It includes the complete input, inference, parsing, recall-check and export code. Default execution is CPU preparation only. GPU calls require restored model weights, the matching vLLM server and `RUN_INFERENCE = True`.

## Inputs and environment

Use the repository’s pinned models, adapter registry, five article sets,20 frozen participant interactions and125 belief-difference records. Follow `Models/README.md` and `Parallel Agent Setup/README.md` first. Keep only one model family loaded on the GPU. Change `YOUR_PATHWAY` only if the repository cannot be discovered from the notebook directory.

In [ ]:
from pathlib import Path
import sys, json, hashlib, importlib.util
YOUR_PATHWAY = "YOUR PATHWAY"
if YOUR_PATHWAY == "YOUR PATHWAY":
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Models/adapter_manifest.json").is_file()), None)
    if ROOT is None:
        raise ValueError("Set YOUR_PATHWAY to the repository clone directory.")
else:
    ROOT = Path(YOUR_PATHWAY).expanduser().resolve()
sys.path[:0] = [str(ROOT / "Experiment Codes/src"), str(ROOT / "Parallel Agent Setup/src")]
RUN_INFERENCE = False
WITH_MANIPULATION_CHECKS = False


## Complete source used by this notebook

The source below is shown verbatim for review. Execution imports these same files from this repository; the following check rejects a stale notebook snapshot. The source is not copied into an independent implementation.

In [ ]:
SOURCE_SHA256 = {'Experiment Codes/src/model_tokenization.py': 'd761ccbab2456b6eb5073f6c3b436fc5b9066f475ffbf7293e54c6eb01f0a7ce', 'Experiment Codes/src/workbench_protocol.py': '056e8d8620a543c481177a41e6bd199222d42db6efd33511d90360d18ce69e75', 'Parallel Agent Setup/src/transport_core.py': 'f2d670c9e9cd6624945afca84fe9f384c58e9de7a19be756813f99c64d395837', 'Parallel Agent Setup/src/workbench_runtime.py': 'f80e133f4f990e26ef88034691e047e3db6b8409913c973a33d15867946b0603', 'Experiment Codes/src/manipulation_checks.py': 'afce713ee7ac933cef3d71cedecba21d06688b9cac2e8891b3a3e1907b91772f', 'Experiment Codes/src/manager_experiment.py': '6cbb7dee28823c6b9951cc71aba072944bef0eab943ed775ffaba119478eebf4', 'Experiment Codes/Mistral/Attempt_01/run_attempt_01.py': '88ee0565d9e2810a7a1d644ecf2c25d88bd61d2d7b8d69ee88501bf5824a670f'}
for relative, expected in SOURCE_SHA256.items():
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected, "Source changed: " + relative
print("Verified complete source files:", len(SOURCE_SHA256))

### Experiment Codes/src/model_tokenization.py

```python
"""Exact local Mistral template and pinned Qwen ChatML; CPU only."""
from pathlib import Path
import ast
import base64
import json
import unicodedata

class NativeTokenizer:
    def __init__(self, root):
        root=Path(root)
        config=json.loads((root/'config/model.json').read_text())
        self.family=config['family']
        folder=(root/config['base_model_dir']).resolve()
        if self.family=='Mistral':
            from tokenizers import Tokenizer
            from jinja2 import Environment, StrictUndefined
            self.encoder=Tokenizer.from_file(str(folder/'tokenizer.json'))
            self.config=json.loads((folder/'tokenizer_config.json').read_text())
            self.template=Environment(undefined=StrictUndefined).from_string(self.config['chat_template'])
        elif self.family=='Qwen':
            import tiktoken
            source=ast.parse((folder/'tokenization_qwen.py').read_text())
            pat=next(ast.literal_eval(n.value) for n in source.body if isinstance(n,ast.Assign)
                     and any(isinstance(t,ast.Name) and t.id=='PAT_STR' for t in n.targets))
            ranks={base64.b64decode(token):int(rank) for line in (folder/'qwen.tiktoken').read_bytes().splitlines()
                   if line for token,rank in [line.split()]}
            special={text:151643+i for i,text in enumerate(['<|endoftext|>','<|im_start|>','<|im_end|>']+
                     [f'<|extra_{i}|>' for i in range(205)])}
            self.encoder=tiktoken.Encoding('pinned_qwen',pat_str=pat,mergeable_ranks=ranks,special_tokens=special)
        else: raise ValueError('Unknown model family')

    def prepare(self, content):
        if self.family=='Mistral':
            def invalid(message):raise ValueError(message)
            text=self.template.render(messages=[{'role':'user','content':content}],
                 bos_token=self.config['bos_token'],eos_token=self.config['eos_token'],
                 raise_exception=invalid,add_generation_prompt=True)
            return text,self.encoder.encode(text,add_special_tokens=False).ids
        encode=lambda text:self.encoder.encode(unicodedata.normalize('NFC',text),allowed_special=set(),disallowed_special=())
        nl=encode('\n')
        ids=([151644]+encode('system')+nl+encode('You are a helpful assistant.')+[151645]+nl+
             [151644]+encode('user')+nl+encode(content)+[151645]+nl+[151644]+encode('assistant')+nl)
        return self.encoder.decode(ids),ids
```

### Experiment Codes/src/workbench_protocol.py

```python
"""Pure input construction: no network, GPU, model generation, or prior responses."""
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import hashlib
import json
import random
import re
import uuid

ROOT = Path(__file__).resolve().parents[1] / "Mistral"
TRANSCRIPT_INTRO = 'The following is a conversation between two other participants. You did not take part in it.'
OWN_ANSWER_INSTRUCTION = 'Before reviewing, state your own answer to the question.'
NEUTRAL = {'P1': 'P1', 'P2': 'P2'}


def read_json(path):
    return json.loads(Path(path).read_text())


def sha_text(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.part')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n')
    temp.replace(path)


def write_json_once(path, value):
    """Create immutable run provenance, or require an identical existing file."""
    path = Path(path)
    data = json.dumps(value, ensure_ascii=False, indent=2) + '\n'
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with path.open('x') as file:
            file.write(data)
    except FileExistsError:
        if path.read_text() != data:
            raise ValueError('Existing run provenance differs; use a new approved run ID: ' + str(path))


def bind_run_sources(run_root, source_paths, root=ROOT):
    root = Path(root)
    files = {str(Path(path).relative_to(root)): hashlib.sha256(Path(path).read_bytes()).hexdigest()
             for path in sorted(map(Path, source_paths))}
    write_json_once(Path(run_root)/'inputs/source_hashes.json', files)
    return files


def seed_for(identifier, base=20261018):
    return (base + int(sha_text(identifier)[:8], 16)) % 2**32


@dataclass(frozen=True)
class AgentInstance:
    """A distinct request context sharing read-only base/adapter tensors.

    There is no mutable conversation, optimizer, or cross-row cache here. Sharing
    a frozen adapter does not share a conversation or update anyone's weights.
    """
    instance_id: str
    adapter_alias: str
    seed: int
    history: tuple = ()


def spawn_instance(key, adapter_alias):
    return AgentInstance(str(uuid.uuid5(uuid.NAMESPACE_URL, 'parallel-workbench:' + key)),
                         adapter_alias, seed_for(key))


def spawn_original_agents(run_id='participant_prepared'):
    return {a: spawn_instance(run_id + ':original:' + a, 'participant_' + a.lower()) for a in 'ABCDE'}


def neutralize(text, actual_to_neutral):
    """Rename explicit speakers once, never by chained replacements.

    Do not replace bare single letters: 'A' may be a word, scientific label or
    article content. Frozen Participant Agent Interaction speaker references use 'Agent A', etc.
    """
    def replace(match):
        actual = match.group(1)
        if actual not in actual_to_neutral:
            raise ValueError('Transcript references an agent outside the pair: ' + actual)
        return actual_to_neutral[actual]
    return re.sub(r'\bAgent\s+([ABCDE])\b', replace, text)


def validate_base_prompt(baseline, attempt):
    """Reject incomplete prompt copies before preparing or sending any request."""
    choice_field = 'VERDICT:' if int(attempt) == 3 else 'ENDORSEMENT:'
    if not any(label in baseline for label in ('YOUR OWN ANSWER:', 'YOUR OWN ANSER:')):
        raise ValueError('Incomplete base prompt: missing own-answer output field')
    if choice_field not in baseline or 'RATIONALE:' not in baseline:
        raise ValueError('Incomplete base prompt: missing choice or rationale output field')
    if 'Return only these three fields.' not in baseline:
        raise ValueError('Incomplete base prompt: missing closing output instruction')
    if int(attempt) == 3 and not all(label in baseline for label in ('P1:', 'P2:', 'Both:', 'None:')):
        raise ValueError('Incomplete verdict prompt: missing category definitions')
    return True


def build_cases(attempt, root=ROOT):
    """100 article/pair/manager cases x two separately generated orientations."""
    root = Path(root)
    attempt = f'{int(attempt):02d}'
    if attempt not in ('01', '02', '03'):
        raise ValueError('Unknown attempt')
    baseline = (root/f'Attempt_{attempt}/prompts/Prompt_Verbatim.txt').read_text()
    validate_base_prompt(baseline, attempt)
    model_config = read_json(root/'config/model.json')
    sessions = read_json(root/'inputs/frozen_sessions.json')
    differences = {(d['article_id'], d['manager'], d['participant']): d
                   for d in read_json(root/'inputs/belief_differences.json')}
    cases = []
    for session in sessions:
        for manager in 'ABCDE':
            base = f'{session["session_id"]}_manager_{manager}'
            for orientation in ['original', 'swapped']:
                pair = session['agents']
                actuals = pair if orientation == 'original' else list(reversed(pair))
                labels = dict(zip(['P1','P2'], actuals))
                actual_to_neutral = {actual: NEUTRAL[p] for p, actual in labels.items()}
                row_id = f'{model_config["family"].lower()}_a{attempt}_{base}_{orientation}'
                instance = spawn_instance(row_id, model_config['manager_alias_prefix'] + manager.lower())
                difference_texts = {
                    p: differences[(session['article_id'], manager, actual)]['text'].format(participant=NEUTRAL[p])
                    for p, actual in labels.items()}
                messages = [{**m, 'speaker': actual_to_neutral[m['actual_agent']],
                             'text': neutralize(m['text'], actual_to_neutral)} for m in session['messages']]
                finals = {p: session['final_message_ids'][actual] for p, actual in labels.items()}
                allowed = ['P1', 'P2'] if attempt != '03' else (
                    ['P1','P2','None'] if session['source_outcome'] == 'Unresolved'
                    else ['P1','P2','Both','None'])
                # Only substitute the explicitly requested manager placeholder.
                # Its neutral role name does not reveal the participant's letter.
                prompt = baseline.replace('{manager}', 'manager')
                sections = [prompt, OWN_ANSWER_INSTRUCTION,
                    'FACTUAL QUESTION\n' + session['question'],
                    'PARTICIPANT IDENTITIES FOR THIS EVALUATION\nP1 and P2 identify the two participants consistently in every supplied material.',
                    'BELIEF DIFFERENCES RELATIVE TO MY LEARNED BELIEFS\n' +
                    '\n'.join(f'Beliefs of {NEUTRAL[p]} ({p}): {difference_texts[p]}' for p in ['P1','P2']),
                    'FINAL MESSAGE IDS\n' + '\n'.join(f'{p} ({NEUTRAL[p]}): {finals[p]}' for p in ['P1','P2'])]
                if attempt in ['02','03']:
                    transcript = '\n\n'.join(f'{m["id"]} | {m["speaker"]} | {m["stage"]}\n{m["text"]}' for m in messages)
                    sections.append(TRANSCRIPT_INTRO + '\n<OTHER_PARTICIPANTS_CONVERSATION>\n' +
                                    transcript + '\n</OTHER_PARTICIPANTS_CONVERSATION>')
                else:
                    sections.append('No interaction transcript is supplied in this attempt. The final message IDs identify the participants\' final messages; their contents are not supplied.')
                if attempt == '03':
                    sections.append('PERMITTED VERDICTS\n' + ', '.join(allowed))
                content = '\n\n'.join(sections)
                cases.append({'row_id': row_id, 'base_row_id': base, 'attempt': attempt,
                    'article_id': session['article_id'], 'article_title': session['article_title'],
                    'session_id': session['session_id'], 'pair': session['pair'], 'manager': manager,
                    'orientation': orientation, 'labels_to_actual': labels, 'neutral_to_actual':
                        {NEUTRAL[p]: a for p, a in labels.items()},
                    'agent_instance': asdict(instance), 'question': session['question'],
                    'components': session['components'], 'belief_differences': difference_texts,
                    'final_message_ids': finals, 'messages': messages, 'allowed_choices': allowed,
                    'source_outcome_offline_only': session['source_outcome'],
                    'baseline_prompt_sha256': sha_text(baseline), 'content': content,
                    'content_sha256': sha_text(content),
                    'source_session_sha256': session['source_session_sha256'],
                    'own_answer_measurement': 'same-call pre-decision answer; exposed to all input materials'})
    validate_cases(cases)
    return cases


def validate_cases(cases):
    if len(cases) != 200 or len({c['row_id'] for c in cases}) != 200:
        raise ValueError('Exactly 200 independently identified calls are required')
    if len({c['agent_instance']['instance_id'] for c in cases}) != 200:
        raise ValueError('Manager contexts must be unique')
    if len({c['agent_instance']['seed'] for c in cases}) != 200:
        raise ValueError('Request seeds unexpectedly collide')
    grouped = {}
    for c in cases:
        if c['agent_instance']['history'] not in [(), []]:
            raise ValueError('Manager context must start empty')
        grouped.setdefault(c['base_row_id'], []).append(c)
        if OWN_ANSWER_INSTRUCTION not in c['content']:
            raise ValueError('Missing own-answer instruction')
        if re.search(r'\bAgent\s+[ABCDE]\b|Manager_Agent_Pair_|P[12]\s*=\s*[ABCDE]\b', c['content']):
            raise ValueError('Original agent identity leaked into manager prompt')
        if c['attempt'] in ['02','03'] and TRANSCRIPT_INTRO not in c['content']:
            raise ValueError('Missing third-party transcript frame')
        for p, mid in c['final_message_ids'].items():
            matches = [m for m in c['messages'] if m['id'] == mid]
            if len(matches) != 1 or matches[0]['actual_agent'] != c['labels_to_actual'][p]:
                raise ValueError('Final-message attribution mismatch')
    if len(grouped) != 100:
        raise ValueError('Expected 100 matched cases')
    for pair in grouped.values():
        if len(pair) != 2 or {c['orientation'] for c in pair} != {'original','swapped'}:
            raise ValueError('Missing independently requested orientation')
        a, b = sorted(pair, key=lambda c:c['orientation'])
        if a['labels_to_actual']['P1'] != b['labels_to_actual']['P2']:
            raise ValueError('Participant positions were not exchanged')
        if a['labels_to_actual']['P2'] != b['labels_to_actual']['P1']:
            raise ValueError('Participant positions were not exchanged')
    return True


def randomized_execution_order(cases, seed=20261018):
    """Random scheduling order; reporting retains adjacent matched orientations."""
    rows = list(cases)
    random.Random(seed).shuffle(rows)
    return rows


class FrozenTokenizer:
    """CPU-only tokenizer; exact local base vocabulary, no weight download."""
    def __init__(self, root=ROOT):
        from model_tokenization import NativeTokenizer
        self.impl = NativeTokenizer(root)
    def serialize(self, content):
        return self.impl.prepare(content)[0]
    def prepare(self, content):
        return self.impl.prepare(content)


def prepare_inputs(attempt, root=ROOT, save=True):
    """Author 200 exact input strings and token IDs. Never generate responses."""
    root = Path(root)
    cases = build_cases(attempt, root)
    tokenizer = FrozenTokenizer(root)
    config = read_json(root/f'Attempt_{int(attempt):02d}/config.json')
    for c in cases:
        c['serialized'], c['input_ids'] = tokenizer.prepare(c['content'])
        c['input_tokens'] = len(c['input_ids'])
        if c['input_tokens'] + config['max_output_tokens'] > read_json(root/'config/model.json')['context_tokens']:
            raise ValueError('Context overflow; inputs must not be truncated: ' + c['row_id'])
    if save:
        dest = root/f'Attempt_{int(attempt):02d}/inputs'
        write_json(dest/'prepared_cases.json', cases)
        write_json(dest/'all_input_texts.json', [c['serialized'] for c in cases])
        write_json(dest/'all_input_token_ids.json', [c['input_ids'] for c in cases])
        write_json(dest/'row_order.json', [c['row_id'] for c in cases])
        write_json(dest/'execution_order.json', [c['row_id'] for c in randomized_execution_order(cases)])
    return cases
```

### Parallel Agent Setup/src/transport_core.py

```python
"""Isolated, resumable requests to one immutable multi-LoRA vLLM server.

This is infrastructure code. It contains no experiment-specific choice rules,
article materials, endorsement policies, or repair loops.
"""
from __future__ import annotations
import asyncio
import codecs
import datetime as dt
import hashlib
import json
import os
import time
from pathlib import Path
from typing import Any
import httpx

def stamp():
    return dt.datetime.now(dt.timezone.utc).isoformat()

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def atomic_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.part')
    tmp.write_text(json.dumps(value,indent=2,ensure_ascii=False)+'\n')
    tmp.replace(path)

def runtime_digest(root):
    root=Path(root)
    server=json.loads((root/'config/server_process.json').read_text())
    return digest({'server_command':server['command'],
      'environment_lock':(root/'config/requirements.lock.txt').read_text(),
      'base_model_manifest':json.loads((root/'config/base_model_manifest.json').read_text()),
      'client_source_sha256':hashlib.sha256(Path(__file__).read_bytes()).hexdigest()})

def request_digest(payload, alias, runtime):
    return digest({'payload':payload,'adapter':alias,'runtime':runtime})

async def journaled_lines(response, path):
    decoder=codecs.getincrementaldecoder('utf-8')()
    pending=''
    with Path(path).open('ab') as journal:
        async for chunk in response.aiter_bytes():
            journal.write(chunk);journal.flush()
            pending+=decoder.decode(chunk)
            while '\n' in pending:
                line,pending=pending.split('\n',1)
                yield line.removesuffix('\r')
        pending+=decoder.decode(b'',final=True)
        if pending:yield pending.removesuffix('\r')

def validate_job(job, aliases, context_limit=16384):
    if job['model'] not in aliases:raise ValueError('Unregistered adapter: '+job['model'])
    ids=job['input_ids']
    if not ids or any(type(x) is not int or x<0 for x in ids):raise ValueError('Explicit token IDs required')
    if len(ids)+job['max_tokens']>context_limit:raise ValueError('Context budget exceeded; truncation is forbidden')
    if not 0<=job['temperature']<=2:raise ValueError('Invalid temperature')
    if not 0<=job['seed']<2**32:raise ValueError('Explicit 32-bit request seed required')
    if not job['call_id'].replace('_','').replace('-','').isalnum():raise ValueError('Unsafe call ID')
    return {
      'model':job['model'],'prompt':ids,'temperature':job['temperature'],
      'max_tokens':job['max_tokens'],'seed':job['seed'],'n':1,
      'top_p':1.0,'top_k':-1,'min_p':0.0,'repetition_penalty':1.0,
      'presence_penalty':0.0,'frequency_penalty':0.0,
      'stop_token_ids':job.get('stop_token_ids',[]),'stop':job.get('stop',[]),'include_stop_str_in_output':True,
      'skip_special_tokens':False,'logprobs':5,'return_tokens_as_token_ids':True,'return_token_ids':True,
      'min_tokens':job.get('min_tokens',0),
      'stream':True,'stream_options':{'include_usage':True}}

async def execute_one(client, semaphore, job, root, aliases, endpoint):
    payload=validate_job(job, aliases)
    runtime=runtime_digest(root)
    signature=request_digest(payload,aliases[job['model']],runtime)
    path=Path(root)/'logs/calls'/(job['call_id']+'.json')
    if path.exists():
        saved=json.loads(path.read_text())
        if saved['request_sha256']!=signature:raise ValueError('A saved call ID has different inputs')
        if saved['status']=='complete':return {**saved,'resumed_from_disk':True}
        raise RuntimeError('Previous call failed; explicit new run ID required: '+job['call_id'])
    path.parent.mkdir(parents=True,exist_ok=True)
    claim=path.with_suffix('.claim')
    fd=os.open(claim,os.O_CREAT|os.O_EXCL|os.O_WRONLY,0o600)
    os.write(fd,str(os.getpid()).encode());os.close(fd)
    queued=time.perf_counter()
    record={'call_id':job['call_id'],'kind':job.get('kind', 'research_call'),
      'status':'queued','queued_utc':stamp(),'request_sha256':signature,
      'request':payload,'adapter':aliases[job['model']],'runtime_sha256':runtime,
      'serialized_input':job['serialized'],'input_tokens':len(job['input_ids']),
      'output_text':'','output_token_ids':[],'logprob_token_ids':[],
      'server_prompt_token_ids':None,'events':[],'usage':None,'received_done':False,
      'finish_reason':None,'stop_reason':None}
    atomic_json(path,record)
    async with semaphore:
        start=time.perf_counter();record.update(status='running',started_utc=stamp(),client_queue_seconds=start-queued)
        atomic_json(path,record)
        print('REQUEST_START', job['call_id'], 'INPUT_TOKENS', len(job['input_ids']), flush=True)
        first=None
        try:
            async with client.stream('POST',endpoint+'/v1/completions',json=payload) as response:
                if response.status_code!=200:
                    await response.aread()
                    raise RuntimeError(f'HTTP {response.status_code}: {response.text[:2000]}')
                async for line in journaled_lines(response,path.with_suffix('.sse')):
                    if not line.startswith('data:'):continue
                    raw=line[5:].lstrip()
                    if raw=='[DONE]':
                        record['received_done']=True;break
                    event=json.loads(raw);record['events'].append(event)
                    if event.get('error'):raise RuntimeError(str(event['error']))
                    if event.get('usage') is not None:record['usage']=event['usage']
                    for choice in event.get('choices',[]):
                        if choice.get('prompt_token_ids') is not None:
                            if record['server_prompt_token_ids'] is not None and record['server_prompt_token_ids']!=choice['prompt_token_ids']:
                                raise RuntimeError('Server prompt token IDs changed within one request')
                            record['server_prompt_token_ids']=choice['prompt_token_ids']
                        record['output_token_ids'].extend(choice.get('token_ids') or [])
                        text=choice.get('text','')
                        if text and first is None:first=time.perf_counter()
                        record['output_text']+=text
                        tokens=(choice.get('logprobs') or {}).get('tokens') or []
                        for token in tokens:
                            if token.startswith('token_id:'):
                                record['logprob_token_ids'].append(int(token.split(':',1)[1]))
                        if choice.get('finish_reason') is not None:record['finish_reason']=choice['finish_reason']
                        if choice.get('stop_reason') is not None:record['stop_reason']=choice['stop_reason']
            if record['usage'] is None:raise RuntimeError('Missing server token usage')
            if record['usage']['prompt_tokens']!=record['input_tokens']:raise RuntimeError('Server input token count mismatch')
            if record['server_prompt_token_ids']!=job['input_ids']:raise RuntimeError('Server prompt token IDs mismatch')
            if not record['received_done'] or record['finish_reason'] not in ['stop','length']:raise RuntimeError('Incomplete completion stream')
            record['output_tokens']=record['usage']['completion_tokens']
            record['output_ids_match_usage']=len(record['output_token_ids'])==record['output_tokens']
            if not record['output_ids_match_usage']:raise RuntimeError('Output token IDs/usage mismatch')
            if record['logprob_token_ids']!=record['output_token_ids']:raise RuntimeError('Logprob and output token IDs mismatch')
            record['status']='complete'
        except asyncio.CancelledError:
            record.update(status='interrupted',error='Request task cancelled; explicit recovery is required')
            raise
        except Exception as exc:
            record.update(status='failed',error=repr(exc))
        finally:
            end=time.perf_counter()
            record.update(finished_utc=stamp(),elapsed_seconds=end-start,
              total_seconds=end-queued,time_to_first_text_seconds=None if first is None else first-start)
            atomic_json(path,record)
            claim.unlink(missing_ok=True)
            print('REQUEST',job['call_id'],record['status'],'INPUT',record['input_tokens'],
                  'OUTPUT',record.get('output_tokens'),'QUEUE_SECONDS',round(record['client_queue_seconds'],3),
                  'GENERATION_SECONDS',round(record['elapsed_seconds'],3),
                  'TOTAL_SECONDS',round(record['total_seconds'],3),flush=True)
    return record

async def monitor_memory(path, stop):
    with Path(path).open('a') as file:
        while not stop.is_set():
            process=await asyncio.create_subprocess_exec('nvidia-smi',
              '--query-gpu=memory.used,memory.free,utilization.gpu','--format=csv,noheader,nounits',
              stdout=asyncio.subprocess.PIPE,stderr=asyncio.subprocess.PIPE)
            out,err=await process.communicate()
            sample={'utc':stamp(),'gpu':out.decode().strip(),'error':err.decode().strip()}
            sample['meminfo']=dict(line.split(':',1) for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith(('MemAvailable:','MemTotal:')))
            file.write(json.dumps(sample)+'\n');file.flush()
            try:await asyncio.wait_for(stop.wait(),timeout=1.0)
            except asyncio.TimeoutError:pass

async def run_batch(jobs, concurrency, root, run_name, aliases, endpoint='http://127.0.0.1:8000'):
    invocation_start=time.perf_counter()
    root=Path(root)
    if len({j['call_id'] for j in jobs})!=len(jobs):raise ValueError('Duplicate call IDs')
    for job in jobs:validate_job(job,aliases)
    runtime=runtime_digest(root)
    for job in jobs:
        path=root/'logs/calls'/(job['call_id']+'.json')
        if path.with_suffix('.claim').exists():raise RuntimeError('Call already claimed; inspect before recovery')
        if path.exists():
            value=json.loads(path.read_text())
            expected=request_digest(validate_job(job,aliases),aliases[job['model']],runtime)
            if value['request_sha256']!=expected:raise ValueError('Saved input/runtime signature changed')
            if value['status']!='complete':raise RuntimeError('Previous call is incomplete; explicit new diagnostic run ID required')
    summary_path=root/'qa'/f'{run_name}.json'
    if summary_path.exists():
        previous=json.loads(summary_path.read_text())
        if previous.get('failed')==0 and previous.get('call_ids')==[j['call_id'] for j in jobs] and previous.get('concurrency')==concurrency:
            saved=[]
            for job in jobs:
                value=json.loads((root/'logs/calls'/(job['call_id']+'.json')).read_text())
                expected=request_digest(validate_job(job,aliases),aliases[job['model']],runtime)
                if value['request_sha256']!=expected:raise ValueError('Existing batch inputs changed')
                saved.append(value)
            if all(value['status']=='complete' for value in saved):
                print('BATCH_RESUMED_WITHOUT_GENERATION',run_name,flush=True)
                return {**previous,'resumed_without_generation':True,
                    'historical_batch_elapsed_seconds':previous['elapsed_seconds'],
                    'current_invocation_seconds':time.perf_counter()-invocation_start},[
                    {**value,'resumed_from_disk':True} for value in saved]
    stop=asyncio.Event();memory_path=root/'logs'/f'{run_name}_memory.jsonl'
    memory_path.parent.mkdir(parents=True,exist_ok=True)
    monitor=asyncio.create_task(monitor_memory(memory_path,stop))
    start=time.perf_counter()
    try:
        async with httpx.AsyncClient(timeout=httpx.Timeout(600.0,connect=30.0),
          limits=httpx.Limits(max_connections=concurrency,max_keepalive_connections=concurrency),trust_env=False) as client:
            before=await client.get(endpoint+'/metrics')
            (root/'logs'/f'{run_name}_metrics_before.txt').write_text(before.text)
            semaphore=asyncio.Semaphore(concurrency)
            tasks=[asyncio.create_task(execute_one(client,semaphore,j,root,aliases,endpoint)) for j in jobs]
            try:
                results=await asyncio.gather(*tasks,return_exceptions=True)
            except BaseException:
                # Never release the endpoint lock while sibling requests remain active.
                for task in tasks:
                    if not task.done():task.cancel()
                await asyncio.gather(*tasks,return_exceptions=True)
                raise
            errors=[{'call_id':job['call_id'],'error':repr(result)}
                    for job,result in zip(jobs,results) if isinstance(result,BaseException)]
            if errors:
                atomic_json(root/'qa'/f'{run_name}_task_errors.json',errors)
                raise RuntimeError('Request task failures drained safely; inspect the saved task errors')
            after=await client.get(endpoint+'/metrics')
            (root/'logs'/f'{run_name}_metrics_after.txt').write_text(after.text)
    finally:
        stop.set();await monitor
    seconds=time.perf_counter()-start
    complete=[r for r in results if r['status']=='complete']
    used=[r for r in complete if not r.get('resumed_from_disk')]
    summary={'run_name':run_name,'utc':stamp(),'concurrency':concurrency,'requests':len(jobs),
      'completed':len(complete),'failed':len(results)-len(complete),
      'elapsed_seconds':seconds,'input_tokens':sum(r['input_tokens'] for r in complete),
      'output_tokens':sum(r.get('output_tokens',0) for r in complete),
      'fresh_output_tokens_per_second':sum(r.get('output_tokens',0) for r in used)/seconds,
      'all_input_counts_match':all(r.get('usage',{}).get('prompt_tokens')==r['input_tokens'] for r in complete),
      'all_output_ids_match_usage':all(r.get('output_ids_match_usage') for r in complete),
      'call_ids':[j['call_id'] for j in jobs], 'resumed_without_generation':False,
      'current_invocation_seconds':time.perf_counter()-invocation_start}
    atomic_json(root/'qa'/f'{run_name}.json',summary)
    print('BATCH_SUMMARY',json.dumps(summary),flush=True)
    return summary,results
```

### Parallel Agent Setup/src/workbench_runtime.py

```python
"""One immutable multi-LoRA server, bounded concurrency, exact per-call audits."""
from __future__ import annotations
from pathlib import Path
import datetime as dt
import fcntl
import hashlib
import json
import os
import re
import shutil

from workbench_protocol import ROOT, read_json, write_json
from transport_core import run_batch, validate_job

ENDPOINT = 'http://127.0.0.1:8000'
LOCK_PATH = Path(__file__).resolve().parents[2] / '.runtime' / 'api_127_0_0_1_8000.lock'


def require_execution_permission(allow_inference):
    if allow_inference is not True:
        raise PermissionError('Prepared only. Set RUN_INFERENCE=True only after user approval to execute.')


def registry(root=ROOT):
    return {a['alias']: a for a in read_json(Path(root)/'config/adapter_registry.json')}


def choose_admission(jobs, root=ROOT):
    if not jobs:
        raise ValueError('No independent ready requests')
    plan = read_json(Path(root)/'config/recommended_runtime.json')
    aliases = registry(root)
    for job in jobs:
        validate_job(job, aliases, plan['max_context_tokens_including_output'])
    maximum = max(len(j['input_ids']) + j['max_tokens'] for j in jobs)
    if all(len(j['input_ids']) <= 584 and j['max_tokens'] <= 128 for j in jobs):
        profile_cap, basis = 32, 'validated short-context capacity'
    elif all(len(j['input_ids']) <= plan['manager_validated_input_tokens'] and j['max_tokens'] <= 600 for j in jobs):
        profile_cap, basis = plan['manager_validated_concurrency'], 'validated manager profile'
    elif all(len(j['input_ids']) <= plan['participant_validated_input_tokens'] and j['max_tokens'] <= 384 for j in jobs):
        profile_cap, basis = plan['participant_validated_concurrency'], 'validated Participant Agent Interaction profile'
    elif all(j['max_tokens'] <= 600 for j in jobs):
        profile_cap, basis = plan['near_context_limit_validated_concurrency'], 'conservative long-context profile'
    else:
        raise ValueError('Output budget is outside the validated capacity profiles')
    cap = min(len(jobs), profile_cap, plan['server_max_sequences'],
              plan['conservative_total_reserved_tokens'] // maximum)
    if cap < 1:
        raise ValueError('No request fits the token reservation budget')
    cap = min(cap, plan['client_default_concurrency'])
    return {'concurrency': cap, 'maximum_reserved_per_request': maximum,
            'total_reserved_at_cap': cap * maximum, 'profile_cap': profile_cap, 'basis': basis}


def validate_run_id(run_id):
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,70}', run_id):
        raise ValueError('Use a unique run ID with only letters, digits, underscore or hyphen')


def initialize_run(folder, root=ROOT):
    """Copy small immutable runtime metadata; never copy model weights."""
    folder, root = Path(folder), Path(root)
    for d in ['config','logs','qa','data','inputs']:
        (folder/d).mkdir(parents=True, exist_ok=True)
    for name in ['server_process.json','requirements.lock.txt','base_model_manifest.json',
                 'adapter_registry.json','recommended_runtime.json']:
        source = (root/'runtime/server_process.json') if name == 'server_process.json' else root/'config'/name
        target = folder/'config'/name
        if target.exists() and target.read_bytes() != source.read_bytes():
            raise ValueError('Cannot resume a run under changed runtime metadata: ' + name)
        if not target.exists():
            shutil.copy2(source, target)


async def preflight_endpoint(root=ROOT):
    """Read-only health check, called only inside the approved execution path."""
    import httpx
    async with httpx.AsyncClient(timeout=20, trust_env=False) as client:
        health = await client.get(ENDPOINT + '/health')
        health.raise_for_status()
        response = await client.get(ENDPOINT + '/v1/models')
        response.raise_for_status()
        models = response.json()
    available = {x['id'] for x in models['data']}
    expected = set(registry(root))
    if not expected <= available:
        raise RuntimeError('Required adapters are not all registered: ' + repr(expected-available))
    return models


async def run_parallel(jobs, run_name, run_root, *, allow_inference=False, root=ROOT):
    """The only inference gateway; calls inside one batch are independent.

    Existing completed calls are resumed from their exact immutable records.
    Incomplete or changed calls fail loudly; no outcome-based retries exist.
    """
    require_execution_permission(allow_inference)
    validate_run_id(run_name)
    admission = choose_admission(jobs, root)
    initialize_run(run_root, root)
    LOCK_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOCK_PATH.open('a+') as lock:
        try:
            fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError('Another coordinator owns the GPU endpoint; wait for that batch to finish') from error
        try:
            lock.seek(0)
            lock.truncate()
            json.dump({'pid':os.getpid(),'run_name':run_name,'utc':dt.datetime.now(dt.timezone.utc).isoformat()}, lock)
            lock.flush()
            models = await preflight_endpoint(root)
            write_json(Path(run_root)/'qa/endpoint_models.json', models)
            write_json(Path(run_root)/'qa'/f'{run_name}_admission.json', admission)
            print('PARALLEL_ADMISSION', json.dumps(admission), flush=True)
            summary, records = await run_batch(jobs, admission['concurrency'], run_root,
                                               run_name, registry(root), ENDPOINT)
            # The batch wall clock includes the slowest call, not the sum of
            # overlapping calls. Cached calls are explicitly identified.
            for record in records:
                print('INSTANCE_RUNTIME', record['call_id'],
                    'INPUT_TOKENS', record['input_tokens'],
                    'OUTPUT_TOKENS', record.get('output_tokens', 'unavailable'),
                    'QUEUE_SECONDS', round(record.get('client_queue_seconds', 0), 6),
                    'GENERATION_SECONDS', round(record.get('elapsed_seconds', 0), 6),
                    'TOTAL_SECONDS', round(record.get('total_seconds', 0), 6),
                    'CACHED', bool(record.get('resumed_from_disk')), flush=True)
            if summary.get('resumed_without_generation'):
                print('HISTORICAL_BATCH_WALL_CLOCK_SECONDS', summary['historical_batch_elapsed_seconds'], flush=True)
            else:
                print('BATCH_WALL_CLOCK_SECONDS', summary['elapsed_seconds'], flush=True)
            print('CURRENT_INVOCATION_SECONDS', summary['current_invocation_seconds'], flush=True)
            return summary, records
        finally:
            fcntl.flock(lock.fileno(), fcntl.LOCK_UN)
```

### Experiment Codes/src/manipulation_checks.py

```python
"""Optional one-shot unaided recall checks, separate from manager decisions."""
from pathlib import Path
import json
from workbench_protocol import ROOT, build_cases, FrozenTokenizer, write_json, write_json_once, seed_for, bind_run_sources
from workbench_runtime import run_parallel, require_execution_permission, validate_run_id

def prepare_checks(root=ROOT):
    tokenizer=FrozenTokenizer(root)
    cases=build_cases(1,root)
    jobs=[]
    for case in cases:
        content='Based on your own knowledge and beliefs, answer every requested part. State your answer concisely.\n\n'+case['question']
        text,ids=tokenizer.prepare(content)
        jobs.append({'row_id':case['row_id'],'model':case['agent_instance']['adapter_alias'],
                     'serialized':text,'input_ids':ids,'seed':seed_for('unaided:'+case['row_id']),
                     'temperature':.6,'max_tokens':160,'kind':'unaided_manipulation_check',
                     'stop_token_ids':json.loads((Path(root)/'config/model.json').read_text())['stop_token_ids']})
    return jobs

async def run_checks(run_id, *, allow_inference=False, root=ROOT):
    require_execution_permission(allow_inference);validate_run_id(run_id)
    folder=Path(root)/'Manipulation_Checks/runs'/run_id
    root=Path(root)
    sources=list((root.parent/'src').glob('*.py'))+list((root.parents[1]/'Parallel Agent Setup/src').glob('*.py'))
    sources += [root/'config/model.json',root/'config/adapter_registry.json',root/'inputs/frozen_sessions.json']
    sources += list(root.glob('Attempt_*/run_attempt_*.py'))
    bind_run_sources(folder,sources,root.parents[1])
    jobs=prepare_checks(root)
    for job in jobs:job['call_id']=run_id+'_'+job['row_id']
    write_json_once(folder/'inputs/jobs.json',jobs)
    timing,records=await run_parallel(jobs,run_id,folder,allow_inference=True,root=root)
    # Human semantic scoring preserves the original response; it never overwrites an answer.
    rubric=json.loads((Path(root).parents[1]/'DATA/assigned_belief_answer_key.json').read_text())
    keys={(x['article_id'],x['manager'],x['component']):x for x in rubric}
    cases={x['row_id']:x for x in build_cases(1,root)}
    byid={r['call_id']:r for r in records};review=[]
    for job in jobs:
        c=cases[job['row_id']];r=byid[job['call_id']]
        review.append({'row_id':c['row_id'],'article_id':c['article_id'],'manager':c['manager'],
            'question':c['question'],'response':r['output_text'],
            'expected_answers':{part:keys[(c['article_id'],c['manager'],part)]['answer'] for part in c['components']},
            'pass_0_or_1':None,'reviewer':'','review_reason':'',
            'input_tokens':r['input_tokens'],'output_tokens':r.get('output_tokens')})
    write_json(folder/'scoring_required.json',review)
    return review
```

### Experiment Codes/src/manager_experiment.py

```python
"""200 independent manager evaluations per attempt; never derive swapped outputs."""
from __future__ import annotations
from collections import Counter
from pathlib import Path
import csv
import json
import re
import sys

from workbench_protocol import (ROOT, build_cases, prepare_inputs, randomized_execution_order,
    read_json, write_json, write_json_once, bind_run_sources, sha_text, seed_for, spawn_instance, validate_cases)
from workbench_runtime import run_parallel, require_execution_permission, validate_run_id


def parse_manager_output(text, attempt, permitted):
    """Retain malformed/forbidden responses; flag them without regenerating."""
    key = 'VERDICT' if int(attempt) == 3 else 'ENDORSEMENT'
    cleaned = text.removesuffix('</s>').removesuffix('<|im_end|>').removesuffix('<|endoftext|>').strip()
    pattern = (r'\AYOUR OWN ANS(?:W?ER):\s*(.*?)\n\s*' + key +
               r':\s*(P1|P2|BOTH|NONE)\s*\n\s*RATIONALE:\s*(.*?)\s*\Z')
    match = re.fullmatch(pattern, cleaned, flags=re.IGNORECASE | re.DOTALL)
    issues = []
    own, choice, rationale = '', '', ''
    if not match:
        issues.append('Output does not follow the required three fields in order')
    else:
        own, raw_choice, rationale = match.groups()
        own, rationale = own.strip(), rationale.strip()
        choice = {'BOTH':'Both','NONE':'None','P1':'P1','P2':'P2'}[raw_choice.upper()]
        if choice not in permitted:
            issues.append('Category is not permitted for this row')
        if not own:
            issues.append('Own answer is empty')
        if not rationale:
            issues.append('Rationale is empty')
        sentence_count = len(re.findall(r'[.!?](?:\s+|$)', rationale))
        if sentence_count < 2 or sentence_count > 5:
            issues.append('Check two-to-five sentence requirement manually')
    return {'own_answer':own, 'choice':choice, 'rationale':rationale,
            'format_valid':bool(match), 'permitted_choice':choice in permitted,
            'review_flags':issues, 'raw_output':text}


def make_jobs(cases, run_id, config):
    validate_run_id(run_id)
    jobs = []
    for c in randomized_execution_order(cases):
        jobs.append({'call_id': run_id + '_' + c['row_id'], 'kind':'manager_experiment',
            'model':c['agent_instance']['adapter_alias'], 'input_ids':c['input_ids'],
            'serialized':c['serialized'], 'temperature':config['temperature'],
            'max_tokens':config['max_output_tokens'], 'seed':c['agent_instance']['seed'],
            'stop_token_ids':config['stop_token_ids']})
    return jobs


def collect_results(cases, records, run_id):
    lookup = {r['call_id']:r for r in records}
    results = []
    for c in cases:
        call_id = run_id + '_' + c['row_id']
        r = lookup[call_id]
        parsed = parse_manager_output(r['output_text'], c['attempt'], c['allowed_choices'])
        if r['status'] != 'complete':
            parsed['review_flags'].append('Inference failed: ' + r.get('error','unknown'))
        if r['finish_reason'] == 'length':
            parsed['review_flags'].append('Output reached the generation limit; retained without repair')
        choice = parsed['choice']
        actual = c['labels_to_actual'].get(choice, choice)
        results.append({'row_id':c['row_id'], 'call_id':call_id, 'attempt':c['attempt'],
            'article_id':c['article_id'], 'article_title':c['article_title'], 'pair':c['pair'],
            'manager':c['manager'], 'orientation':c['orientation'], 'base_row_id':c['base_row_id'],
            'instance_id':c['agent_instance']['instance_id'],
            'adapter':c['agent_instance']['adapter_alias'], 'seed':c['agent_instance']['seed'],
            'P1':c['labels_to_actual']['P1'], 'P2':c['labels_to_actual']['P2'],
            'question':c['question'], 'source_outcome':c['source_outcome_offline_only'],
            **parsed, 'actual_choice':actual,
            'readable_choice':f'{choice}={actual}' if choice in ['P1','P2'] else choice,
            'status':r['status'], 'finish_reason':r['finish_reason'],
            'input_tokens':r['input_tokens'], 'output_tokens':r.get('output_tokens'),
            'queue_seconds':r.get('client_queue_seconds'),
            'generation_seconds':r.get('elapsed_seconds'), 'total_seconds':r.get('total_seconds'),
            'time_to_first_text_seconds':r.get('time_to_first_text_seconds'),
            'input_sha256':sha_text(c['serialized']), 'raw_call_path':'logs/calls/'+call_id+'.json'})
    return results


def position_comparison(rows):
    grouped = {}
    for r in rows:
        grouped.setdefault(r['base_row_id'], {})[r['orientation']] = r
    result = []
    for key, pair in grouped.items():
        a, b = pair['original'], pair['swapped']
        valid = all(r['status']=='complete' and r['format_valid'] and r['permitted_choice'] for r in [a,b])
        result.append({'base_case':key, 'article_id':a['article_id'], 'pair':a['pair'], 'manager':a['manager'],
            'original_choice':a['choice'], 'original_actual_choice':a['actual_choice'],
            'swapped_choice':b['choice'], 'swapped_actual_choice':b['actual_choice'],
            'same_actual_choice':int(a['actual_choice']==b['actual_choice']) if valid else None,
            'both_valid':valid, 'independent_model_calls':2})
    return result


def export_results(rows, summary, destination):
    """Export completed model responses and their recorded metadata."""
    # Pure-Python wheels travel with the package. No pip install or server
    # environment change is needed merely to export a future spreadsheet.
    for wheel in sorted((ROOT/'vendor/wheels').glob('*.whl')):
        if str(wheel) not in sys.path:
            sys.path.insert(0,str(wheel))
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    write_json(destination/'results.json', rows)
    comparisons = position_comparison(rows)
    wb = Workbook()
    wb.remove(wb.active)

    def sheet(name, data, columns):
        ws = wb.create_sheet(name)
        ws.append(columns)
        for row in data:
            ws.append([json.dumps(row.get(k),ensure_ascii=False) if isinstance(row.get(k),(list,dict)) else row.get(k) for k in columns])
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = ws.dimensions
        for c in ws[1]:
            c.font = Font(bold=True, color='FFFFFF')
            c.fill = PatternFill('solid', fgColor='17365D')
        for col in ws.columns:
            heading = col[0].value
            ws.column_dimensions[col[0].column_letter].width = 60 if heading.endswith(('rationale','own_answer','question')) else 22
            for cell in col[1:]:
                cell.alignment = Alignment(vertical='top', wrap_text=True)
        return ws

    columns = ['article_id','pair','manager','orientation','P1','P2','own_answer','readable_choice','rationale',
               'input_tokens','output_tokens','queue_seconds','generation_seconds','total_seconds',
               'time_to_first_text_seconds','source_outcome','status','finish_reason','format_valid',
               'permitted_choice','review_flags','instance_id','call_id','seed','input_sha256']
    sheet('Long_Results', rows, columns)
    wide = {}
    for r in rows:
        key = (r['article_id'], r['pair'], r['orientation'])
        w = wide.setdefault(key, {k:r[k] for k in ['article_id','pair','orientation','P1','P2','source_outcome']})
        for field in ['own_answer','readable_choice','rationale','input_tokens','output_tokens','total_seconds']:
            w[r['manager']+'_'+field] = r[field]
    wide_cols = ['article_id','pair','orientation','P1','P2','source_outcome'] + [a+'_'+f for a in 'ABCDE'
        for f in ['own_answer','readable_choice','rationale','input_tokens','output_tokens','total_seconds']]
    sheet('Wide_Results', list(wide.values()), wide_cols)
    sheet('Position_Comparison', comparisons, list(comparisons[0]))
    sheet('Method_Notes', [{'note':s} for s in [
        '200 independent model calls; swapped orientations are not copied or rewritten responses.',
        'Fresh request contexts share frozen weights. There is no additional training or carried conversation.',
        'Own answer is generated first in the same call, after exposure to all supplied materials.',
        'Original Participant Agent Interaction outcomes are reference metadata; they are not manager correctness scores.',
        'Malformed, disallowed and length-limited responses remain visible. No outcome-based repairs or quotas.',
        'Per-call totals overlap in parallel; use batch elapsed_seconds for overall wall-clock duration.',
        'Different seeds and numerical batch effects can also contribute to differences between orientations.'
    ]], ['note'])
    wb.save(destination/'Results.xlsx')
    with (destination/'Tokens_and_Runtimes.csv').open('w',newline='') as f:
        fields = ['call_id','input_tokens','output_tokens','queue_seconds','generation_seconds','total_seconds','time_to_first_text_seconds','status']
        writer = csv.DictWriter(f,fields,extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)
    with (destination/'All_Responses.txt').open('w') as f:
        for r in rows:
            f.write(r['call_id']+'\n'+r['raw_output']+'\n\n')


async def run_attempt(attempt, run_id, *, allow_inference=False, root=ROOT):
    require_execution_permission(allow_inference)
    root = Path(root)
    attempt = f'{int(attempt):02d}'
    validate_run_id(run_id)
    cases = prepare_inputs(attempt, root, save=False)
    # Runtime identities are specific to this run, while input text is unchanged.
    for c in cases:
        c['agent_instance'] = spawn_instance(run_id+':'+c['row_id'], c['agent_instance']['adapter_alias']).__dict__
    validate_cases(cases)
    config = read_json(root/f'Attempt_{attempt}/config.json')
    config['stop_token_ids'] = read_json(root/'config/model.json')['stop_token_ids']
    jobs = make_jobs(cases, run_id, config)
    run_root = root/f'Attempt_{attempt}/runs'/run_id
    sources = [root/f'Attempt_{attempt}/prompts/Prompt_Verbatim.txt',
        root/f'Attempt_{attempt}/config.json', root/'inputs/frozen_sessions.json',
        root/'inputs/belief_differences.json', root/'config/adapter_registry.json']
    sources += list((root.parent/'src').glob('*.py'))
    sources += list((root.parents[1]/'Parallel Agent Setup/src').glob('*.py'))
    sources += [root/'config/model.json', root/f'Attempt_{attempt}/run_attempt_{attempt}.py', root.parent/'run.py']
    sources += list((root/'vendor/wheels').glob('*.whl'))
    bind_run_sources(run_root, sources, root.parents[1])
    write_json_once(run_root/'inputs/all_input_texts.json', [c['serialized'] for c in cases])
    write_json_once(run_root/'inputs/all_input_token_ids.json', [c['input_ids'] for c in cases])
    write_json_once(run_root/'inputs/cases.json', cases)
    summary, records = await run_parallel(jobs, run_id, run_root, allow_inference=True, root=root)
    rows = collect_results(cases, records, run_id)
    summary['source_outcomes_are_not_manager_truth_scores'] = True
    summary['decision_counts'] = dict(Counter(r['choice'] or 'unparsed' for r in rows))
    summary['cases_with_review_flags'] = sum(bool(r['review_flags']) for r in rows)
    destination = root/f'Attempt_{attempt}/deliverables'/run_id
    export_results(rows, summary, destination)
    export_exact_io(cases, records, run_id, destination)
    write_json(run_root/'data/results.json', rows)
    return rows, summary


def export_exact_io(cases, records, run_id, destination):
    destination = Path(destination)
    by_id = {r['call_id']:r for r in records}
    token_rows = []
    with (destination/'All_Exact_Input_and_Output.txt').open('w') as file:
        for case in cases:
            record = by_id[run_id+'_'+case['row_id']]
            file.write(case['row_id']+'\nINPUT\n'+case['serialized']+'\nOUTPUT\n'+record['output_text']+'\n\n')
            token_rows.append({'row_id':case['row_id'],'input_ids':case['input_ids'],
                'output_ids':record['output_token_ids'],'input_tokens':record['input_tokens'],
                'output_tokens':record.get('output_tokens')})
    write_json(destination/'All_Input_and_Output_Token_IDs.json', token_rows)
```

### Experiment Codes/Mistral/Attempt_01/run_attempt_01.py

```python
"""Run Mistral Attempt 1 from input construction through spreadsheet export.

Default: validate the complete 200-call batch on CPU, without model generation.
With --execute: send 200 independent requests through the local vLLM server.
Each original and swapped orientation uses a distinct request context and seed.
Both generated responses are retained independently.
"""
from pathlib import Path
import argparse
import asyncio
import json
import sys

REPOSITORY = Path(__file__).resolve().parents[3]
sys.path[:0] = [str(REPOSITORY / "Experiment Codes/src"),
               str(REPOSITORY / "Parallel Agent Setup/src")]
from workbench_protocol import prepare_inputs, validate_cases
from manager_experiment import run_attempt
from manipulation_checks import prepare_checks, run_checks

FAMILY = "Mistral"
ATTEMPT = 1
WORKSPACE = REPOSITORY / "Experiment Codes" / FAMILY


def prepare(with_manipulation_checks=False):
    """Build and tokenize every full input; validate identities and context limits."""
    cases = prepare_inputs(ATTEMPT, WORKSPACE, save=False)
    validate_cases(cases)
    config = json.loads((WORKSPACE / f"Attempt_{ATTEMPT:02d}/config.json").read_text())
    report = {
        "model": FAMILY, "attempt": ATTEMPT, "manager_calls": len(cases),
        "independent_original_calls": sum(c["orientation"] == "original" for c in cases),
        "independent_swapped_calls": sum(c["orientation"] == "swapped" for c in cases),
        "maximum_input_tokens": max(c["input_tokens"] for c in cases),
        "reserved_output_tokens": config["max_output_tokens"],
        "manipulation_check_calls": len(prepare_checks(WORKSPACE)) if with_manipulation_checks else 0,
        "inference_executed": False,
    }
    print(json.dumps(report, indent=2))
    return cases


async def execute(run_id, *, with_manipulation_checks=False):
    """Optional one-shot recall, followed by parallel manager generation and export.

    Recall runs in separate contexts and is not inserted into manager prompts.
    The first response from each recall call is saved for separate semantic
    scoring.
    """
    if with_manipulation_checks:
        checks = await run_checks(run_id + "_recall", allow_inference=True, root=WORKSPACE)
        print("Unaided responses retained for semantic scoring:", len(checks))
    rows, timing = await run_attempt(ATTEMPT, run_id, allow_inference=True, root=WORKSPACE)
    print("Generated manager responses:", len(rows))
    print("Batch wall seconds:", timing.get("elapsed_seconds"))
    print("Deliverables:", str((WORKSPACE / f"Attempt_{ATTEMPT:02d}/deliverables" / run_id).relative_to(REPOSITORY)))
    return rows, timing


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--run-id", help="A new identifier for a new batch; required with --execute.")
    parser.add_argument("--execute", action="store_true", help="Generate with the restored local vLLM server.")
    parser.add_argument("--with-manipulation-checks", action="store_true", help="Add 200 separate one-shot unaided checks before the manager batch.")
    args = parser.parse_args(argv)
    if args.execute and not args.run_id:
        parser.error("--execute requires --run-id; use a new ID for a new experiment.")
    prepare(args.with_manipulation_checks)
    if args.execute:
        asyncio.run(execute(args.run_id, with_manipulation_checks=args.with_manipulation_checks))


if __name__ == "__main__":
    main()
```

## Prepare and inspect

This validates all original and swapped inputs and their context budgets without generating a response. Participant identities, beliefs, transcript speakers and final-message IDs swap together. Every orientation receives a separate seed and request context.

In [ ]:
entrypoint = ROOT / 'Experiment Codes/Mistral/Attempt_01/run_attempt_01.py'
spec = importlib.util.spec_from_file_location("complete_mistral_attempt_01", entrypoint)
runner = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = runner
spec.loader.exec_module(runner)
RUN_ID = "mistral_attempt01_001"
cases = runner.prepare(WITH_MANIPULATION_CHECKS)
print("First matched pair:", [(c["row_id"], c["labels_to_actual"]) for c in cases[:2]])


## Execute and export

Run only after reviewing the inputs and starting the matching server. Optional manipulation checks ask one unaided question per context and retain the first answer; they do not provide articles, retry until correct, or change the manager response. The manager batch is separately generated in fresh contexts.

In [ ]:
if RUN_INFERENCE:
    rows, timing = await runner.execute(RUN_ID, with_manipulation_checks=WITH_MANIPULATION_CHECKS)
    print("Stored responses:", len(rows))
else:
    print("CPU preparation complete; no model calls were made.")


## Outputs and interpretation

Look in `Experiment Codes/Mistral/Attempt_01/deliverables/<run-id>/` for `Results.xlsx`, `All_Responses.txt`, `results.json`, `Tokens_and_Runtimes.csv`, exact input/output text and token IDs. `Position_Comparison` in the workbook compares the two independently generated orientations. Both independently generated responses are retained in the outputs.